In [1]:
import pandas as pd
import numpy as np
import joblib

In [2]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split

In [3]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier

In [4]:
from sklearn.metrics import (
accuracy_score,
precision_score,
recall_score,
f1_score,
roc_auc_score,
matthews_corrcoef
)

In [5]:
# Load dataset
data = load_breast_cancer()

X = pd.DataFrame(data.data, columns=data.feature_names)

y = pd.Series(data.target)

In [6]:
# Train/Test split
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=42,stratify=y)

In [7]:
# Save test data
test_df = X_test.copy()
test_df["target"] = y_test.values
test_df.to_csv("test_data.csv", index=False)

models = {
    "Logistic Regression": LogisticRegression(max_iter=10000),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "KNN": KNeighborsClassifier(n_neighbors=5),
    "Naive Bayes": GaussianNB(),
    "Random Forest": RandomForestClassifier(
        n_estimators=100,
        random_state=42
    )
}

results = []

for i, (name, model) in enumerate(models.items(), start=1):

    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)

    print("Iteration", i, "done")

    if hasattr(model, "predict_proba"):
        y_prob = model.predict_proba(X_test)[:, 1]
    else:
        y_prob = y_pred

    accuracy = accuracy_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_prob)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    mcc = matthews_corrcoef(y_test, y_pred)

    results.append([
        name,
        accuracy,
        auc,
        precision,
        recall,
        f1,
        mcc
    ])

    filename = name.lower().replace(" ", "_") + ".pkl"
    joblib.dump(model, filename)

results_df = pd.DataFrame(
    results,
    columns=[
        "Model",
        "Accuracy",
        "AUC",
        "Precision",
        "Recall",
        "F1",
        "MCC"
    ]
)

print(results_df)

results_df.to_csv("model_metrics.csv", index=False)

Iteration 1 done
Iteration 2 done
Iteration 3 done
Iteration 4 done
Iteration 5 done
                 Model  Accuracy       AUC  Precision    Recall        F1  \
0  Logistic Regression  0.964912  0.995370   0.959459  0.986111  0.972603   
1        Decision Tree  0.912281  0.915675   0.955882  0.902778  0.928571   
2                  KNN  0.912281  0.955853   0.942857  0.916667  0.929577   
3          Naive Bayes  0.938596  0.987765   0.945205  0.958333  0.951724   
4        Random Forest  0.956140  0.993717   0.958904  0.972222  0.965517   

        MCC  
0  0.924518  
1  0.817412  
2  0.813927  
3  0.867553  
4  0.905447  
